<a href="https://colab.research.google.com/github/UXI-create/-_First-Project/blob/main/%E6%97%85%E9%81%8A%E8%A8%88%E7%95%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 🛑 第一步：環境準備與靜音
# ==========================================
from IPython.utils import io
import os
import warnings
import base64

with io.capture_output() as captured:
    !pip install -q geopy pandas ipywidgets requests

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import ipywidgets as widgets
import pandas as pd
from datetime import datetime, timedelta, date
from IPython.display import display, clear_output, HTML
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests

# ==========================================
# 🎨 視覺美化設定
# ==========================================
app_header = widgets.HTML("""
<div style="background: linear-gradient(135deg, #11998e, #38ef7d); padding: 20px; border-radius: 10px; text-align: center; color: white; margin-bottom: 20px; box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
    <h1 style="margin: 0; font-size: 28px; font-weight: bold; color: white;">✈️ Travel Vibe Pro Max</h1>
    <p style="margin: 5px 0 0 0; font-size: 16px; opacity: 0.9;">多重行程建檔 ✕ 雲端記憶庫 ✕ 完整規劃</p>
</div>
""")

def style_dataframe(df):
    return df.style.set_table_styles([
        {'selector': 'thead th', 'props': [('background-color', '#11998e'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '10px')]},
        {'selector': 'tbody td', 'props': [('text-align', 'center'), ('padding', '8px'), ('border-bottom', '1px solid #eeeeee')]},
        {'selector': 'tbody tr:hover', 'props': [('background-color', '#eafff5')]}
    ]).hide(axis="index").format({'💰 預算': 'NT$ {:,.0f}'})

# ==========================================
# 💾 微型資料庫 (存放多個行程)
# ==========================================
trip_database = {}

# 全域 UI 樣式設定 (解決擠壓問題)
ui_style = {'description_width': 'initial'}

# ==========================================
# ⛅ 模組一：智慧天氣與打包引擎
# ==========================================
visa_rules = {
    "🇯🇵 日本": "免簽證 (建議先填 Visit Japan Web)", "🇰🇷 韓國": "免簽證", "🇹🇭 泰國": "需申請簽證/免簽",
    "🇭🇰 香港": "需申請『預辦入境登記』或『台胞證』", "🇲🇴 澳門": "免簽證", "🇨🇳 中國大陸": "需持有『台胞證』", "🇹🇼 台灣": "身分證/健保卡"
}

def get_weather(country_name):
    try:
        geolocator = Nominatim(user_agent="vibe_weather_app")
        loc = geolocator.geocode(country_name.split(" ")[1] if " " in country_name else country_name)
        if not loc: return None
        url = f"https://api.open-meteo.com/v1/forecast?latitude={loc.latitude}&longitude={loc.longitude}&current_weather=true"
        res = requests.get(url).json()
        return res['current_weather']
    except: return None

def generate_smart_packing_list(days, country):
    packing = {"🪪 證件與金錢": [], "⛅ 天氣與穿搭推薦": [], "🔌 電子與雜物": []}
    if "台灣" in country: packing["🪪 證件與金錢"].extend(["身分證", "健保卡", "現金 / 信用卡"])
    else:
        packing["🪪 證件與金錢"].extend(["護照 (效期6個月以上)", f"⚠️ 簽證: {visa_rules.get(country, '請查詢')}", "機票/訂房憑證"])
        if "中國" in country: packing["🪪 證件與金錢"].append("📱 支付寶/微信支付")
        else: packing["🪪 證件與金錢"].append("外幣現金 / 海外信用卡")

    weather_data = get_weather(country)
    if weather_data:
        temp, code = weather_data['temperature'], weather_data['weathercode']
        is_raining = code >= 50
        packing["⛅ 天氣與穿搭推薦"].append(f"📡 AI 氣象站：氣溫約 {temp}°C，{'🌧️ 預計有雨' if is_raining else '☀️ 天氣晴朗'}")
        if temp < 15: packing["⛅ 天氣與穿搭推薦"].extend(["🧣 發熱衣/保暖內衣", "🧥 羽絨外套或厚大衣"])
        elif temp > 28: packing["⛅ 天氣與穿搭推薦"].extend(["🕶️ 太陽眼鏡", "🧴 防曬乳", "🩳 輕薄透氣短袖"])
        if is_raining: packing["⛅ 天氣與穿搭推薦"].extend(["☔ 摺疊傘 / 輕便雨衣", "🥾 防水鞋套或好乾的鞋子"])

    packing["⛅ 天氣與穿搭推薦"].extend([f"上衣/內外褲 x {days+1} 件", f"襪子 x {days+1} 雙", "好走鞋子"])
    packing["🔌 電子與雜物"].extend(["手機", "充電器/線", "常備藥品", "行動電源"])
    return packing

# ==========================================
# 🗓️ 模組二：自動估算與跨日追蹤引擎
# ==========================================
class SmartPlanner:
    def __init__(self):
        self.itinerary = []
        self.current_time = datetime.now()
        self.geolocator = Nominatim(user_agent="travel_vibe_ultimate_router")
        self.search_cache = {}

    def search_locations(self, city, keyword):
        query = f"{keyword}, {city}" if city else keyword
        try:
            locs = self.geolocator.geocode(query, exactly_one=False, limit=5)
            if not locs: locs = self.geolocator.geocode(keyword, exactly_one=False, limit=5)
            results = {}
            if locs:
                for loc in locs: results[loc.address] = {"name": keyword, "lat": loc.latitude, "lng": loc.longitude}
            return results
        except: return {}

    def estimate_time(self, loc1, loc2, mode):
        if not loc1 or not loc2: return 20
        dist_km = geodesic((loc1['lat'], loc1['lng']), (loc2['lat'], loc2['lng'])).km
        speed = 4 if "步行" in mode else 15 if "公車" in mode else 30
        return max(3, int(((dist_km * 1.3) / speed) * 60))

    def format_time_str(self, mins):
        if mins < 60: return f"{mins} 分"
        return f"{mins // 60} 小時 {mins % 60} 分" if mins % 60 > 0 else f"{mins // 60} 小時"

    def add_stop(self, place_address, stay_mins, trans_mode, cost, note, start_datetime=None):
        place_data = self.search_cache.get(place_address)
        if not place_data: return False, "系統錯誤"

        stay_time = timedelta(minutes=stay_mins)
        if not self.itinerary:
            if start_datetime: self.current_time = start_datetime
            arrival_time = self.current_time
            transport_text = "✨ 出發"
        else:
            last_stop = self.itinerary[-1]
            trans_mins = self.estimate_time(last_stop['coords'], place_data, trans_mode)
            arrival_time = self.current_time + timedelta(minutes=trans_mins)
            transport_text = f"{trans_mode} (約 {self.format_time_str(trans_mins)})"

        departure_time = arrival_time + stay_time

        self.itinerary.append({
            "seq": len(self.itinerary) + 1,
            "arrive": arrival_time.strftime("%m/%d %H:%M"),
            "transport": transport_text,
            "name": place_data['name'],
            "note": note,
            "stay": self.format_time_str(stay_mins),
            "cost": cost,
            "depart": departure_time.strftime("%m/%d %H:%M"),
            "coords": place_data
        })
        self.current_time = departure_time
        return True, ""

    def reset(self):
        self.itinerary = []

# ==========================================
# 📱 模組三：UI 介面 (修復擠壓問題)
# ==========================================
box_layout = widgets.Layout(padding='15px', border='1px solid #e0e0e0', border_radius='10px', margin='10px 0')
highlight_box_layout = widgets.Layout(padding='15px', border='2px solid #11998e', border_radius='10px', margin='10px 0', background_color='#f4fff9')

# --- 行李分頁 ---
dest_dropdown = widgets.Dropdown(options=list(visa_rules.keys()), value='🇯🇵 日本', description='✈️ 目的地:', style=ui_style)
days_input = widgets.IntText(value=5, description='📅 天數:', layout=widgets.Layout(width='150px'), style=ui_style)
pack_btn = widgets.Button(description=" ⛅ 預測天氣並生成", button_style='success', layout=widgets.Layout(width='180px'))
pack_out = widgets.Output()

def on_pack_click(b):
    with pack_out:
        clear_output()
        print(f"📡 正在分析 {dest_dropdown.value} 當地氣候...")
        result = generate_smart_packing_list(days_input.value, dest_dropdown.value)
        clear_output()
        html_str = f"<h3 style='color: #11998e;'>🎒 【 {dest_dropdown.value} {days_input.value}天 】 必備清單</h3>"
        for cat, items in result.items():
            html_str += f"<div style='background: #f8f9fa; padding: 10px; border-left: 4px solid #38ef7d; margin-bottom: 10px;'>"
            html_str += f"<h4 style='margin-top: 0;'>{cat}</h4><ul style='list-style-type: none; padding-left: 10px;'>"
            for item in items: html_str += f"<li style='margin-bottom: 5px;'>🔲 {item}</li>"
            html_str += "</ul></div>"
        display(HTML(html_str))
pack_btn.on_click(on_pack_click)
tab_pack = widgets.VBox([widgets.HBox([dest_dropdown, days_input, pack_btn]), pack_out], layout=box_layout)

# --- 行程分頁 ---
planner = SmartPlanner()

# 檔案管理區 UI
trip_name_input = widgets.Text(description='📁 行程命名:', placeholder='例: 2024 東京跨年', layout=widgets.Layout(width='280px'), style=ui_style)
save_trip_btn = widgets.Button(description=" 💾 儲存目前行程", button_style='success')
saved_trips_dropdown = widgets.Dropdown(options=['無暫存紀錄'], description='📂 歷史紀錄:', layout=widgets.Layout(width='280px'), style=ui_style)
load_trip_btn = widgets.Button(description=" 📖 讀取", button_style='info')
new_trip_btn = widgets.Button(description=" 📄 開新空白行程", button_style='danger')
file_msg_out = widgets.Output()

# 🌟 這裡修復了時間選單被擠壓的問題
start_date_picker = widgets.DatePicker(description='📅 起點日期:', value=date.today(), layout=widgets.Layout(width='200px'), style=ui_style)
start_hour = widgets.Dropdown(options=[f"{i:02d}" for i in range(24)], value='08', description='時間:', layout=widgets.Layout(width='120px'), style=ui_style)
start_min = widgets.Dropdown(options=[f"{i:02d}" for i in range(0, 60, 5)], value='00', description='分:', layout=widgets.Layout(width='120px'), style=ui_style)

city_input = widgets.Text(value='東京', description='🌍 城市:', layout=widgets.Layout(width='180px'), style=ui_style)
keyword_input = widgets.Text(description='🔍 找景點/機場:', placeholder='例: 成田機場', layout=widgets.Layout(width='220px'), style=ui_style)
search_btn = widgets.Button(description=" 🔎 搜尋", button_style='info')

place_dropdown = widgets.Dropdown(options=['請先搜尋地點...'], description='📍 確認地點:', disabled=True, layout=widgets.Layout(width='450px'), style=ui_style)
stay_input = widgets.IntText(value=120, description='⏱️ 停留(分):', layout=widgets.Layout(width='160px'), style=ui_style)
trans_mode = widgets.Dropdown(options=['🚶 步行', '🚌 公車/地鐵', '🚗 計程車/開車', '✈️ 搭飛機/高鐵'], value='🚌 公車/地鐵', description='交通:', layout=widgets.Layout(width='180px'), style=ui_style)

cost_input = widgets.IntText(value=0, description='💰 預算(元):', layout=widgets.Layout(width='160px'), style=ui_style)
note_input = widgets.Text(description='📝 備註:', placeholder='航班號 / 備註資訊', layout=widgets.Layout(width='320px'), style=ui_style)
add_btn = widgets.Button(description=" ➕ 加入行程", button_style='primary', disabled=True)
export_btn = widgets.Button(description=" 📥 匯出 CSV", button_style='success')
plan_out = widgets.Output()

def update_plan_view():
    with plan_out:
        clear_output()
        if not planner.itinerary:
            display(HTML("<p style='color: gray;'>📭 目前為空白行程。設定日期時間後，開始搜尋景點吧！</p>"))
            return

        df = pd.DataFrame(planner.itinerary)
        total_cost = df['cost'].sum()
        display_df = df[['arrive', 'transport', 'name', 'stay', 'note', 'cost', 'depart']].copy()
        display_df.columns = ['抵達時間', '前往此站', '📍 地點', '⏱️ 停留', '📝 航班/備註', '💰 預算', '離開時間']

        display(style_dataframe(display_df))
        display(HTML(f"<h3 style='text-align: right; color: #11998e;'>💵 總花費: NT$ {total_cost:,.0f} </h3>"))

# 檔案管理邏輯
def on_save_click(b):
    with file_msg_out:
        clear_output()
        name = trip_name_input.value.strip()
        if not name:
            print("❌ 請先輸入行程名稱才能儲存！")
            return
        if not planner.itinerary:
            print("❌ 行程是空的，沒有東西可以存！")
            return
        import copy
        trip_database[name] = copy.deepcopy(planner.itinerary)
        saved_trips_dropdown.options = list(trip_database.keys())
        saved_trips_dropdown.value = name
        print(f"✅ 成功儲存行程：【{name}】！")

def on_load_click(b):
    with file_msg_out:
        clear_output()
        name = saved_trips_dropdown.value
        if name not in trip_database:
            print("❌ 找不到該行程紀錄。")
            return
        import copy
        planner.itinerary = copy.deepcopy(trip_database[name])
        start_date_picker.disabled = True
        start_hour.disabled = True
        start_min.disabled = True
        trip_name_input.value = name
        print(f"📖 成功讀取行程：【{name}】")
    update_plan_view()

def on_new_trip_click(b):
    planner.reset()
    trip_name_input.value = ""
    start_date_picker.disabled = False
    start_hour.disabled = False
    start_min.disabled = False
    with file_msg_out:
        clear_output()
        print("📄 已開啟全新空白行程，請重新設定出發日期！")
    update_plan_view()

save_trip_btn.on_click(on_save_click)
load_trip_btn.on_click(on_load_click)
new_trip_btn.on_click(on_new_trip_click)

def on_search_click(b):
    with plan_out:
        clear_output()
        display(HTML(f"<p>📡 雷達掃描中...</p>"))
    results = planner.search_locations(city_input.value, keyword_input.value)
    with plan_out:
        clear_output()
        if not results:
            place_dropdown.options = ['找不到地點']
            place_dropdown.disabled, add_btn.disabled = True, True
        else:
            planner.search_cache.update(results)
            place_dropdown.options = list(results.keys())
            place_dropdown.disabled, add_btn.disabled = False, False
            display(HTML(f"<p style='color: #27ae60;'>✅ 找到地點，請確認並填寫航班備註後點擊「加入」！</p>"))
            update_plan_view()

def on_add_click(b):
    if place_dropdown.disabled: return

    start_dt = None
    if not planner.itinerary:
        time_str = f"{start_hour.value}:{start_min.value}"
        start_dt = datetime.combine(start_date_picker.value, datetime.strptime(time_str, "%H:%M").time())

    planner.add_stop(place_dropdown.value, stay_input.value, trans_mode.value, cost_input.value, note_input.value, start_dt)

    keyword_input.value = ""
    cost_input.value = 0
    note_input.value = ""
    place_dropdown.options = ['請先搜尋下一個地點...']
    place_dropdown.disabled, add_btn.disabled = True, True
    start_date_picker.disabled = True
    start_hour.disabled = True
    start_min.disabled = True

    update_plan_view()

def on_export_click(b):
    if not planner.itinerary: return
    df = pd.DataFrame(planner.itinerary)
    export_df = df[['arrive', 'transport', 'name', 'stay', 'note', 'cost', 'depart']].copy()
    export_df.columns = ['抵達時間 (月/日 時:分)', '交通方式', '地點', '停留時間', '航班號/備註', '預估花費', '離開時間 (月/日 時:分)']

    csv_data = export_df.to_csv(index=False, encoding='utf-8-sig')
    b64 = base64.b64encode(csv_data.encode('utf-8-sig')).decode()

    trip_name = trip_name_input.value if trip_name_input.value else "未命名行程"
    href = f"""
    <div style="margin-top: 15px;">
        <a download="Travel_Vibe_{trip_name}.csv" href="data:text/csv;base64,{b64}" target="_blank"
           style="padding: 10px 20px; background-color: #11998e; color: white; text-decoration: none; border-radius: 5px; font-weight: bold; box-shadow: 0 2px 4px rgba(0,0,0,0.2);">
           💾 點此下載【Travel_Vibe_{trip_name}.csv】
        </a>
    </div>
    """
    with plan_out:
         display(HTML(href))

search_btn.on_click(on_search_click)
add_btn.on_click(on_add_click)
export_btn.on_click(on_export_click)

# 佈局分組
file_row1 = widgets.HBox([trip_name_input, save_trip_btn])
file_row2 = widgets.HBox([saved_trips_dropdown, load_trip_btn, new_trip_btn])
file_section = widgets.VBox([widgets.HTML("<b>📂 歷史行程檔案管理</b>"), file_row1, file_row2, file_msg_out], layout=highlight_box_layout)

row0 = widgets.HBox([start_date_picker, start_hour, start_min])
row1 = widgets.HBox([city_input, keyword_input, search_btn])
row2 = widgets.HBox([place_dropdown])
row3 = widgets.HBox([stay_input, trans_mode, cost_input])
row4 = widgets.HBox([note_input, add_btn, export_btn])

tab_plan = widgets.VBox([
    file_section,
    widgets.HTML("<b>📍 STEP 1: 設定行程起點日期與時間</b>"), row0, widgets.HTML("<hr>"),
    widgets.HTML("<b>📍 STEP 2: 搜尋與加入行程</b>"), row1, row2, row3, row4,
    widgets.HTML("<hr>"), plan_out
], layout=box_layout)

app_tabs = widgets.Tab(children=[tab_plan, tab_pack])
app_tabs.set_title(0, '🗓️ 智慧行程與檔案庫')
app_tabs.set_title(1, '🧳 天氣與打包管家')

clear_output()
display(app_header)
display(app_tabs)